In [5]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
from datasets import load_from_disk
import re
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


In [6]:
# Load your trained model
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v1" 

print("Loading trained model for BLEU evaluation...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},  # Use GPU 0
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("✅ Model loaded successfully for BLEU evaluation!")

Loading trained model for BLEU evaluation...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.51it/s]


Tokenizer vocab size: 128256


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


✅ Model loaded successfully for BLEU evaluation!


In [ ]:
def test_model(patient_id, query, max_length=2000):
    """Test model with mental health query"""
    # Format exactly like training data
    messages = [
        {
            "role": "system",
            "content": """You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions."""
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.1,              
            top_p=0.95,                 
            repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

print("✅ test_model function defined!")

✅ test_model function defined!


In [8]:
# Test the function quickly
test_query = "I'm feeling anxious about my upcoming job interview."
test_response = test_model("UID_TEST", test_query)
print("🧪 Test Response:")
print(test_response)
print("="*50)
print("✅ Model working correctly - ready for BLEU evaluation!")

🧪 Test Response:
It's completely normal to feel anxious about a job interview, especially if it's for a position you really want. Here are some suggestions that might help you manage your anxiety and prepare for the interview:

1. Preparation is key: Start by researching the company and the position you're applying for. Understand their values, mission, and what they look for in candidates. This knowledge will give you a better idea of how to approach the interview and showcase your skills and experiences.

2. Practice answering common interview questions: Look up common interview questions related to your field or industry. Practice answering them out loud or with a friend. This will help you feel more confident and familiar with the types of questions that may be asked.

3. Take care of yourself: Make sure you're getting enough sleep, eating well, and exercising regularly. Taking care of your physical health can have a positive impact on your mental well-being and overall performance

In [9]:
# Download required NLTK data
nltk.download('punkt', quiet=True)

True

In [10]:
def parse_dataset_sample(sample):
    """Parse your dataset sample to extract query and ground truth response"""
    return {
        'query': sample['query'],
        'ground_truth': sample['response'],
        'instruction': sample['instruction']
    }

def calculate_bleu_scores(predictions, references):
    """Calculate BLEU scores for mental health conversations"""
    
    # Tokenize predictions and references
    pred_tokens = []
    ref_tokens = []
    
    for pred, ref in zip(predictions, references):
        # Clean and tokenize (preserve mental health specific terms)
        pred_clean = re.sub(r'[^\w\s\(\)\[\],\.!?;:]', '', pred.lower()).split()
        ref_clean = re.sub(r'[^\w\s\(\)\[\],\.!?;:]', '', ref.lower()).split()
        
        pred_tokens.append(pred_clean)
        ref_tokens.append([ref_clean])  # BLEU expects list of references
    
    # Calculate individual BLEU scores
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_4_scores = []
    
    for pred, ref in zip(pred_tokens, ref_tokens):
        # Individual sentence BLEU scores
        bleu_1 = sentence_bleu(ref, pred, weights=(1, 0, 0, 0))
        bleu_2 = sentence_bleu(ref, pred, weights=(0.5, 0.5, 0, 0))
        bleu_4 = sentence_bleu(ref, pred, weights=(0.25, 0.25, 0.25, 0.25))
        
        bleu_1_scores.append(bleu_1)
        bleu_2_scores.append(bleu_2)
        bleu_4_scores.append(bleu_4)
    
    # Corpus-level BLEU
    corpus_bleu_1 = corpus_bleu(ref_tokens, pred_tokens, weights=(1, 0, 0, 0))
    corpus_bleu_2 = corpus_bleu(ref_tokens, pred_tokens, weights=(0.5, 0.5, 0, 0))
    corpus_bleu_4 = corpus_bleu(ref_tokens, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25))
    
    return {
        'individual_bleu_1': bleu_1_scores,
        'individual_bleu_2': bleu_2_scores,
        'individual_bleu_4': bleu_4_scores,
        'corpus_bleu_1': corpus_bleu_1,
        'corpus_bleu_2': corpus_bleu_2,
        'corpus_bleu_4': corpus_bleu_4,
        'avg_bleu_1': sum(bleu_1_scores) / len(bleu_1_scores),
        'avg_bleu_2': sum(bleu_2_scores) / len(bleu_2_scores),
        'avg_bleu_4': sum(bleu_4_scores) / len(bleu_4_scores)
    }

print("✅ BLEU evaluation functions ready for mental health dataset!")

✅ BLEU evaluation functions ready for mental health dataset!


In [11]:
# Load test dataset and run BLEU evaluation
print("📊 Loading mental health test dataset for BLEU evaluation...")

# Load your test dataset
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-mental-chat-16k_v1")
test_dataset = dataset["test"]

# Take a subset for initial testing (adjust size as needed)
test_subset = test_dataset.select(range(50))  # Start with 50 samples

print(f"🧪 Evaluating BLEU on {len(test_subset)} test samples...")

predictions = []
references = []
queries = []

for i, sample in enumerate(test_subset):
    # Parse the sample using your dataset structure
    parsed = parse_dataset_sample(sample)
    query = parsed['query']
    ground_truth = parsed['ground_truth']
    
    # Generate prediction using your model
    prediction = test_model("UID_TEST", query)
    
    predictions.append(prediction)
    references.append(ground_truth)
    queries.append(query)
    
    if (i + 1) % 10 == 0:
        print(f"✅ Processed {i + 1}/{len(test_subset)} samples...")

print(f"🎯 Generated {len(predictions)} predictions for BLEU evaluation")

📊 Loading mental health test dataset for BLEU evaluation...
🧪 Evaluating BLEU on 50 test samples...
✅ Processed 10/50 samples...
✅ Processed 20/50 samples...
✅ Processed 30/50 samples...
✅ Processed 40/50 samples...
✅ Processed 50/50 samples...
🎯 Generated 50 predictions for BLEU evaluation


In [12]:
# Calculate and display BLEU scores
print("📊 CALCULATING BLEU SCORES FOR MENTAL HEALTH CONVERSATIONS...")
print("="*70)

bleu_results = calculate_bleu_scores(predictions, references)

print("🎯 MENTAL HEALTH CONVERSATION BLEU RESULTS:")
print(f"   Corpus BLEU-1: {bleu_results['corpus_bleu_1']:.4f}")
print(f"   Corpus BLEU-2: {bleu_results['corpus_bleu_2']:.4f}")
print(f"   Corpus BLEU-4: {bleu_results['corpus_bleu_4']:.4f}")

print(f"\n📊 Average BLEU Scores:")
print(f"   Average BLEU-1: {bleu_results['avg_bleu_1']:.4f}")
print(f"   Average BLEU-2: {bleu_results['avg_bleu_2']:.4f}")
print(f"   Average BLEU-4: {bleu_results['avg_bleu_4']:.4f}")

# Show sample comparisons
print(f"\n🔍 SAMPLE MENTAL HEALTH CONVERSATION COMPARISONS:")
for i in range(min(3, len(predictions))):
    print(f"\n{'='*60}")
    print(f"SAMPLE {i+1}:")
    print(f"Query: {queries[i][:100]}...")
    print(f"\nGround Truth: {references[i][:150]}...")
    print(f"\nPrediction: {predictions[i][:150]}...")
    print(f"\nBLEU-4 Score: {bleu_results['individual_bleu_4'][i]:.4f}")

print("="*70)

📊 CALCULATING BLEU SCORES FOR MENTAL HEALTH CONVERSATIONS...
🎯 MENTAL HEALTH CONVERSATION BLEU RESULTS:
   Corpus BLEU-1: 0.4736
   Corpus BLEU-2: 0.2889
   Corpus BLEU-4: 0.1407

📊 Average BLEU Scores:
   Average BLEU-1: 0.4324
   Average BLEU-2: 0.2633
   Average BLEU-4: 0.1259

🔍 SAMPLE MENTAL HEALTH CONVERSATION COMPARISONS:

SAMPLE 1:
Query: Throughout our counseling session, I would like your help with managing stress-related challenges in...

Ground Truth: Managing stress-related challenges can be a difficult task, but with the right strategies and support, it is possible to find relief. It seems like hi...

Prediction: Managing stress can be a challenging task, especially when it becomes overwhelming and starts affecting various aspects of your life. It's great that ...

BLEU-4 Score: 0.1140

SAMPLE 2:
Query: (sighing deeply) I've been feeling overwhelmed since I left the hospital. I've been trying to get ba...

Ground Truth: (leaning forward, offering a sympathetic smile) I ca

In [13]:
# Additional analysis for mental health specific evaluation
print("🧠 MENTAL HEALTH SPECIFIC BLEU ANALYSIS:")
print("="*50)

# Analyze BLEU distribution
import numpy as np

bleu_4_scores = bleu_results['individual_bleu_4']
print(f"📊 BLEU-4 Score Distribution:")
print(f"   Min:    {min(bleu_4_scores):.4f}")
print(f"   Max:    {max(bleu_4_scores):.4f}")
print(f"   Mean:   {np.mean(bleu_4_scores):.4f}")
print(f"   Median: {np.median(bleu_4_scores):.4f}")
print(f"   Std:    {np.std(bleu_4_scores):.4f}")

# Categorize performance
excellent = [s for s in bleu_4_scores if s >= 0.4]
good = [s for s in bleu_4_scores if 0.2 <= s < 0.4]
fair = [s for s in bleu_4_scores if 0.1 <= s < 0.2]
poor = [s for s in bleu_4_scores if s < 0.1]

print(f"\n🎯 Performance Categories:")
print(f"   Excellent (≥0.4): {len(excellent)}/{len(bleu_4_scores)} ({len(excellent)/len(bleu_4_scores)*100:.1f}%)")
print(f"   Good (0.2-0.4):   {len(good)}/{len(bleu_4_scores)} ({len(good)/len(bleu_4_scores)*100:.1f}%)")
print(f"   Fair (0.1-0.2):   {len(fair)}/{len(bleu_4_scores)} ({len(fair)/len(bleu_4_scores)*100:.1f}%)")
print(f"   Poor (<0.1):      {len(poor)}/{len(bleu_4_scores)} ({len(poor)/len(bleu_4_scores)*100:.1f}%)")

print("="*50)

🧠 MENTAL HEALTH SPECIFIC BLEU ANALYSIS:
📊 BLEU-4 Score Distribution:
   Min:    0.0223
   Max:    0.2386
   Mean:   0.1259
   Median: 0.1278
   Std:    0.0511

🎯 Performance Categories:
   Excellent (≥0.4): 0/50 (0.0%)
   Good (0.2-0.4):   5/50 (10.0%)
   Fair (0.1-0.2):   30/50 (60.0%)
   Poor (<0.1):      15/50 (30.0%)


In [14]:
import numpy as np
from rouge_score import rouge_scorer

def calculate_rouge_scores(predictions, references):
    """Calculate ROUGE scores for mental health conversations"""
    
    # Initialize ROUGE scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred, ref in zip(predictions, references):
        # Calculate ROUGE scores for this pair
        scores = scorer.score(ref, pred)
        
        # Extract F1 scores (most commonly reported)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    return {
        'rouge1_scores': rouge1_scores,
        'rouge2_scores': rouge2_scores,
        'rougeL_scores': rougeL_scores,
        'avg_rouge1': np.mean(rouge1_scores),
        'avg_rouge2': np.mean(rouge2_scores),
        'avg_rougeL': np.mean(rougeL_scores),
        'std_rouge1': np.std(rouge1_scores),
        'std_rouge2': np.std(rouge2_scores),
        'std_rougeL': np.std(rougeL_scores)
    }

print("✅ ROUGE evaluation functions ready!")

✅ ROUGE evaluation functions ready!


In [15]:
# Calculate ROUGE scores using the same predictions and references from BLEU
print("📊 CALCULATING ROUGE SCORES FOR MENTAL HEALTH CONVERSATIONS...")
print("="*70)

rouge_results = calculate_rouge_scores(predictions, references)

print("🎯 MENTAL HEALTH CONVERSATION ROUGE RESULTS:")
print(f"   ROUGE-1 (Unigram overlap): {rouge_results['avg_rouge1']:.4f} ± {rouge_results['std_rouge1']:.4f}")
print(f"   ROUGE-2 (Bigram overlap):  {rouge_results['avg_rouge2']:.4f} ± {rouge_results['std_rouge2']:.4f}")
print(f"   ROUGE-L (Longest sequence): {rouge_results['avg_rougeL']:.4f} ± {rouge_results['std_rougeL']:.4f}")

print(f"\n📊 ROUGE Score Distributions:")
print(f"   ROUGE-1: Min={min(rouge_results['rouge1_scores']):.4f}, Max={max(rouge_results['rouge1_scores']):.4f}")
print(f"   ROUGE-2: Min={min(rouge_results['rouge2_scores']):.4f}, Max={max(rouge_results['rouge2_scores']):.4f}")
print(f"   ROUGE-L: Min={min(rouge_results['rougeL_scores']):.4f}, Max={max(rouge_results['rougeL_scores']):.4f}")

print("="*70)

📊 CALCULATING ROUGE SCORES FOR MENTAL HEALTH CONVERSATIONS...
🎯 MENTAL HEALTH CONVERSATION ROUGE RESULTS:
   ROUGE-1 (Unigram overlap): 0.5610 ± 0.1141
   ROUGE-2 (Bigram overlap):  0.2221 ± 0.0603
   ROUGE-L (Longest sequence): 0.2593 ± 0.0572

📊 ROUGE Score Distributions:
   ROUGE-1: Min=0.2357, Max=0.6774
   ROUGE-2: Min=0.0671, Max=0.3368
   ROUGE-L: Min=0.1357, Max=0.3746


In [16]:
# Combined BLEU + ROUGE Analysis
print("🔍 COMBINED BLEU + ROUGE ANALYSIS:")
print("="*60)

# Compare BLEU and ROUGE performance
print("📊 METRIC COMPARISON:")
print(f"   BLEU-4:  {bleu_results['avg_bleu_4']:.4f}")
print(f"   ROUGE-1: {rouge_results['avg_rouge1']:.4f}")
print(f"   ROUGE-2: {rouge_results['avg_rouge2']:.4f}")
print(f"   ROUGE-L: {rouge_results['avg_rougeL']:.4f}")

# Correlation analysis
correlation_r1_bleu = np.corrcoef(rouge_results['rouge1_scores'], bleu_4_scores)[0,1]
correlation_r2_bleu = np.corrcoef(rouge_results['rouge2_scores'], bleu_4_scores)[0,1]
correlation_rL_bleu = np.corrcoef(rouge_results['rougeL_scores'], bleu_4_scores)[0,1]

print(f"\n🔗 METRIC CORRELATIONS:")
print(f"   ROUGE-1 vs BLEU-4: {correlation_r1_bleu:.4f}")
print(f"   ROUGE-2 vs BLEU-4: {correlation_r2_bleu:.4f}")
print(f"   ROUGE-L vs BLEU-4: {correlation_rL_bleu:.4f}")

print("="*60)

🔍 COMBINED BLEU + ROUGE ANALYSIS:
📊 METRIC COMPARISON:
   BLEU-4:  0.1259
   ROUGE-1: 0.5610
   ROUGE-2: 0.2221
   ROUGE-L: 0.2593

🔗 METRIC CORRELATIONS:
   ROUGE-1 vs BLEU-4: 0.7803
   ROUGE-2 vs BLEU-4: 0.8969
   ROUGE-L vs BLEU-4: 0.8355


In [17]:
# Performance categorization for ROUGE scores
print("🏆 ROUGE PERFORMANCE CATEGORIZATION:")
print("="*50)

def categorize_rouge_performance(scores, metric_name):
    excellent = [s for s in scores if s >= 0.6]
    good = [s for s in scores if 0.4 <= s < 0.6]
    fair = [s for s in scores if 0.2 <= s < 0.4]
    poor = [s for s in scores if s < 0.2]
    
    total = len(scores)
    print(f"\n{metric_name} Categories:")
    print(f"   Excellent (≥0.6): {len(excellent)}/{total} ({len(excellent)/total*100:.1f}%)")
    print(f"   Good (0.4-0.6):   {len(good)}/{total} ({len(good)/total*100:.1f}%)")
    print(f"   Fair (0.2-0.4):   {len(fair)}/{total} ({len(fair)/total*100:.1f}%)")
    print(f"   Poor (<0.2):      {len(poor)}/{total} ({len(poor)/total*100:.1f}%)")

categorize_rouge_performance(rouge_results['rouge1_scores'], "ROUGE-1")
categorize_rouge_performance(rouge_results['rouge2_scores'], "ROUGE-2")
categorize_rouge_performance(rouge_results['rougeL_scores'], "ROUGE-L")

print("="*50)

🏆 ROUGE PERFORMANCE CATEGORIZATION:

ROUGE-1 Categories:
   Excellent (≥0.6): 26/50 (52.0%)
   Good (0.4-0.6):   19/50 (38.0%)
   Fair (0.2-0.4):   5/50 (10.0%)
   Poor (<0.2):      0/50 (0.0%)

ROUGE-2 Categories:
   Excellent (≥0.6): 0/50 (0.0%)
   Good (0.4-0.6):   0/50 (0.0%)
   Fair (0.2-0.4):   36/50 (72.0%)
   Poor (<0.2):      14/50 (28.0%)

ROUGE-L Categories:
   Excellent (≥0.6): 0/50 (0.0%)
   Good (0.4-0.6):   0/50 (0.0%)
   Fair (0.2-0.4):   43/50 (86.0%)
   Poor (<0.2):      7/50 (14.0%)


In [18]:
# Best and worst samples analysis with both BLEU and ROUGE
print("🔍 DETAILED SAMPLE ANALYSIS (BLEU + ROUGE):")
print("="*70)

# Find samples with highest and lowest combined scores
combined_scores = []
for i in range(len(bleu_4_scores)):
    # Weighted combination of metrics
    combined = (bleu_4_scores[i] * 0.3 + 
                rouge_results['rouge1_scores'][i] * 0.25 + 
                rouge_results['rouge2_scores'][i] * 0.25 + 
                rouge_results['rougeL_scores'][i] * 0.2)
    combined_scores.append(combined)

best_idx = combined_scores.index(max(combined_scores))
worst_idx = combined_scores.index(min(combined_scores))

print(f"🏆 BEST PERFORMING SAMPLE (Combined Score: {max(combined_scores):.4f}):")
print(f"   BLEU-4: {bleu_4_scores[best_idx]:.4f}")
print(f"   ROUGE-1: {rouge_results['rouge1_scores'][best_idx]:.4f}")
print(f"   ROUGE-2: {rouge_results['rouge2_scores'][best_idx]:.4f}")
print(f"   ROUGE-L: {rouge_results['rougeL_scores'][best_idx]:.4f}")
print(f"\nQuery: {queries[best_idx]}")
print(f"\nGround Truth: {references[best_idx][:200]}...")
print(f"\nPrediction: {predictions[best_idx][:200]}...")

print(f"\n⚠️ WORST PERFORMING SAMPLE (Combined Score: {min(combined_scores):.4f}):")
print(f"   BLEU-4: {bleu_4_scores[worst_idx]:.4f}")
print(f"   ROUGE-1: {rouge_results['rouge1_scores'][worst_idx]:.4f}")
print(f"   ROUGE-2: {rouge_results['rouge2_scores'][worst_idx]:.4f}")
print(f"   ROUGE-L: {rouge_results['rougeL_scores'][worst_idx]:.4f}")
print(f"\nQuery: {queries[worst_idx]}")
print(f"\nGround Truth: {references[worst_idx][:200]}...")
print(f"\nPrediction: {predictions[worst_idx][:200]}...")

print("="*70)

🔍 DETAILED SAMPLE ANALYSIS (BLEU + ROUGE):
🏆 BEST PERFORMING SAMPLE (Combined Score: 0.3941):
   BLEU-4: 0.2386
   ROUGE-1: 0.6678
   ROUGE-2: 0.3368
   ROUGE-L: 0.3566

Query: There have been moments when I've contemplated whether life is worth living. Thoughts of self-harm or suicide have crossed my mind during particularly dark periods. While I haven't acted on these thoughts, they scare me, and I want to find healthier ways to cope.

Ground Truth: It's important to acknowledge the courage it takes to reach out and share your experiences. It sounds like you've been going through some really challenging times, and I want you to know that there ar...

Prediction: It's important to acknowledge the strength and courage it takes to reach out for support when you're feeling this way. It sounds like you've been experiencing some difficult thoughts and feelings, and...

⚠️ WORST PERFORMING SAMPLE (Combined Score: 0.1157):
   BLEU-4: 0.0223
   ROUGE-1: 0.2584
   ROUGE-2: 0.0671
   ROUGE-L: 0